In [1]:
%load_ext autoreload
%autoreload 2
from pathlib import Path
import sys

working_directory = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in (working_directory, *working_directory.parents)
     if (path / "pyproject.toml").is_file() and (path / "src").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Open this notebook from within the project directory.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
PROJECT_ROOT.relative_to(PROJECT_ROOT).as_posix()

'.'

In [3]:
BENCHMARK_CONFIG = PROJECT_ROOT / "configs" / "benchmark.yaml"
OUTPUT_PATH = PROJECT_ROOT / "configs" / "benchmark_targets.yaml"

print(BENCHMARK_CONFIG.relative_to(PROJECT_ROOT).as_posix())
print(OUTPUT_PATH.relative_to(PROJECT_ROOT).as_posix())

configs/benchmark.yaml
configs/benchmark_targets.yaml


In [4]:
import yaml

from src.data.loader import load_dataset
from src.data.targets import select_benchmark_targets

In [5]:
with open(BENCHMARK_CONFIG, "r", encoding="utf-8") as file:
    config = yaml.safe_load(file)["benchmark"]

In [6]:
datasets = config["datasets"]
n_targets = config["targets_per_dataset"]
seed = config["target_selection_seed"]

In [7]:
BOLD = "\033[1m"
CYAN = "\033[36m"
GREEN = "\033[32m"
YELLOW = "\033[33m"
RESET = "\033[0m"

selections = {}
for dataset_name in datasets:
    dataset = load_dataset(dataset_name)
    targets = select_benchmark_targets(dataset=dataset, n_targets=n_targets, seed=seed)
    selections[dataset_name] = targets
    
    shape_str = str(dataset.values.shape)
    
    print(
        f"{CYAN}{dataset_name:<15}{RESET} "
        f"shape: {YELLOW}{shape_str:<15}{RESET} - "
        f"targets: {GREEN}{targets}{RESET}")

ETTh1           shape: (17420, 7)      - targets: ['OT', 'HULL', 'MULL']
Weather         shape: (52695, 21)     - targets: ['OT', 'rh (%)', 'SWDR (W/m�)']
Electricity     shape: (26304, 321)    - targets: ['146', '22', '163']
Traffic         shape: (17544, 862)    - targets: ['T683', 'T472', 'T855']
Exchange        shape: (7588, 8)       - targets: ['4', 'OT', '0']
Solar           shape: (52560, 137)    - targets: ['solar_125', 'solar_115', 'solar_27']


In [9]:
with open(OUTPUT_PATH, "w", encoding="utf-8") as file:
    yaml.safe_dump({"targets": selections}, file, sort_keys=False)
print(f"Saved in: {YELLOW}{OUTPUT_PATH.relative_to(PROJECT_ROOT).as_posix()}")

Saved in: configs/benchmark_targets.yaml
